# Analyze feature contributions in held-out test set -- compare to exploratory analyses in train set

- Salience
- LMG Feature Importance

## Set Up

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH, SEED
from src.utils import (
    BEST_PARAM_DICT,
    get_test_df_path,
    col_config_sensitivity,
    export_data,
)
from src.sig import prep_train_test_model
from src.residual_func import get_saliency_map
from src.lmg_imp import get_confirm_lmg


import joblib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

model_dir = BASE_PATH / "models"
df_dir = BASE_PATH / "data/test_data/cleaned"
res_export_path = BASE_PATH / "results/test"

## Salience

In [ ]:
for impact_type in ["self", "parent"]:
    model_params = BEST_PARAM_DICT["SQT"][impact_type]
    model_SQT = joblib.load(model_dir / f"SQT_{impact_type}.joblib")
    coef_df = model_SQT.get_coef_df()
    coef_df["stage_feat"] = coef_df["stage"] + "_" + coef_df["feature"]
    coef_df = coef_df.drop(["stage", "feature"], axis=1)
    coef_df = coef_df.set_index("stage_feat")
    coef_df = coef_df.rename(columns={"coef": f"coef_{impact_type}"})
    test_df_path = get_test_df_path(
        df_dir,
        llm_model=model_params["llm_model"],
        agg_type=model_params["agg_type"],
        desc_seg_type=model_params["desc_seg_type"],
        impact_type=impact_type,
        temp=model_params["temp"],
    )
    test_df = pd.read_parquet(test_df_path)
    model_params["impact_type"] = impact_type
    summary = get_saliency_map(
        config_dict=model_params,
        model=model_SQT,
        df=test_df,
        coef_df=coef_df,
    )
    display(summary)
    export_data(
        data_to_export=summary,
        export_path=res_export_path / f"tables/saliency/{impact_type}.tsv",
        remove_existing=True,
    )

## LMG

In [ ]:
train_data_dir = BASE_PATH / "data"
test_data_dir = df_dir
model_dir = BASE_PATH / "models"
all_perm_dir = all_perm_dir = BASE_PATH / "results/perm_res/preds"
START_PERM = 0
END_PERM = 1000
SHARED_DICT = {
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}
IMPACT_TYPES = [
    "parent",
    #'self' #no sig results, do not include
]
SETUPS = ["SQT", "QT"]

In [ ]:
# res_dict = {}
# for impact_type in IMPACT_TYPES:
#     print(f"{impact_type}...")
#     for setup in SETUPS:
#         print(f"\tSetup: {setup}")
#         model_params = BEST_PARAM_DICT[setup][impact_type]
#         X_train, y_train, X_test, y_test, model = prep_train_test_model(
#             selected_params=model_params,
#             impact_type=impact_type,
#             train_data_dir=train_data_dir,
#             test_data_dir=test_data_dir,
#             shared_dict=SHARED_DICT,
#         )
#         model_config_dict = {
#             **{
#                 k: v
#                 for k, v in model_params.items()
#                 if k not in ["outcome_sub_list", "llm_model", "temp"]
#             },
#             "quest_cols": col_config_sensitivity[f"{impact_type}_quest"],
#             "impact_type": impact_type,
#             **SHARED_DICT,
#         }
#         final_df = get_confirm_lmg(
#             parent_perm_dir=all_perm_dir,
#             X_train=X_train,
#             y_train=y_train,
#             X_test=X_test,
#             y_test=y_test,
#             model_config_dict=model_config_dict,
#             start_perm=START_PERM,
#             end_perm=END_PERM,
#         )
#         res_dict[impact_type] = final_df
#         display(final_df)
#         export_data(
#             data_to_export=final_df,
#             export_path=res_export_path / f"tables/LMG/{impact_type}_{setup}.tsv",
#             remove_existing=True,
#         )

## Aggregate

In [ ]:
N_TOP = 10
all_dfs_list = []
top_contributors = set()
train_cv_lmg_dir = BASE_PATH / "results/feat_cont/cv"
for impact_type in IMPACT_TYPES:
    for setup in SETUPS:
        ## Load train CV results
        train_lmg_series = pd.read_csv(
            train_cv_lmg_dir / f"{impact_type}_{setup}.tsv", sep="\t"
        ).set_index("outcome")["avg_mean"]
        ## Load test results
        test_lmg_series = pd.read_csv(
            res_export_path / f"tables/LMG/{impact_type}_{setup}.tsv", sep="\t"
        ).set_index("outcome")["avg_increase"]
        # Rename cols
        train_lmg_series = train_lmg_series.rename(f"train_{impact_type}_{setup}")
        test_lmg_series = test_lmg_series.rename(f"test_{impact_type}_{setup}")
        # Get top N
        top_train = train_lmg_series.nlargest(N_TOP).index
        top_contributors.update(top_train)
        top_test = test_lmg_series.nlargest(N_TOP).index
        top_contributors.update(top_test)
        # Save cols
        all_dfs_list += [train_lmg_series, test_lmg_series]
## Create large df
all_lmg_df = pd.concat(all_dfs_list, axis=1)
all_lmg_df_sub = all_lmg_df.loc[list(top_contributors)]

# Normalized df (values are % of total contribution)
all_lmg_df_norm = all_lmg_df / all_lmg_df.sum(axis=0)
assert np.allclose(all_lmg_df_norm.sum(axis=0), 1)
all_lmg_df_sub_norm = all_lmg_df_norm.loc[list(top_contributors)]

Make heatmap

In [ ]:
def plot_contributions(plot_df, title):
    # Stack Train/test_parent/self_QT/SQT label
    plot_df.columns = [c.replace("_", "\n") for c in plot_df.columns]
    fig, ax = plt.subplots(figsize=(7, 8))
    ax = sns.heatmap(
        plot_df,
        annot=True,
        fmt=".1%",
        cmap=sns.diverging_palette(10, 130, as_cmap=True),
        center=0,  # white/yellow midpoint at 0
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)  # keep stacked labels upright
    plt.xlabel("Model")
    plt.ylabel("Outcome")
    plt.title(title)
    plt.tight_layout()

In [ ]:
# TODO: Check label order w/ Antonio
# Fix outcome labels
outcome_df = pd.read_excel(
    BASE_PATH / "data/complete_raw/80_Depression_Outcomes.xlsx", index_col=0
)
outcome_df["temp_label"] = [f"S{i+1}" for i in range(len(outcome_df))]
label_map = dict(zip(outcome_df["temp_label"], outcome_df["Outcome"]))

plot_df = all_lmg_df_sub.rename(index=label_map)
plot_df_norm = all_lmg_df_sub_norm.rename(index=label_map)

plot_contributions(plot_df, title="Avg unique R^2 contribution")
plot_contributions(plot_df_norm, title="Avg contribution to total R^2 difference")